# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
import pandas as pd
import numpy as np

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "content_refresh_anonymized.csv"
]
data_path = next((p for p in candidates if os.path.exists(p)), candidates[0])
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows from {data_path}")

# Build feature matrix
feature_cols = [
    "search_volume", "competition", "cpc", "avg_position",
    "impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
    "ctr", "engagement_rate", "scroll_rate", "days_since_last_update"
]

X = df[feature_cols].copy()
X["log_impressions_90d"] = np.log1p(X["impressions_90d"])
X["log_search_volume"] = np.log1p(X["search_volume"])
X = X.fillna(X.median(numeric_only=True))

print("Feature Vector Shape:", X.shape)
print("Preview of engineered features:")
print(X[["log_impressions_90d", "log_search_volume", "ctr", "avg_position"]].head())


Loaded 30000 rows from data/raw/content_refresh_anonymized.csv
Feature Vector Shape: (30000, 14)
Preview of engineered features:
   log_impressions_90d  log_search_volume   ctr  avg_position
0             8.243808           2.397895  0.76          10.6
1             9.636980           4.510860  0.05          20.3
2             9.440023           0.000000  0.09          36.5
3             9.371779           2.397895  0.49           6.2
4             9.859588           0.000000  0.13          44.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# Feature notes: Verify timing and availability before decision
feature_summary = []
for col in X.columns:
    feature_summary.append({
        "feature": col,
        "available_before_decision": True,
        "missing_count": df[col].isnull().sum() if col in df else 0,
        "dtype": str(X[col].dtype)
    })
notes_df = pd.DataFrame(feature_summary)
print(notes_df)


                   feature  available_before_decision  missing_count    dtype
0            search_volume                       True           2468  float64
1              competition                       True           2468  float64
2                      cpc                       True           2468  float64
3             avg_position                       True              0  float64
4          impressions_90d                       True              0    int64
5               clicks_90d                       True              0    int64
6             sessions_90d                       True              0    int64
7    days_with_impressions                       True              0    int64
8                      ctr                       True              0  float64
9          engagement_rate                       True              0  float64
10             scroll_rate                       True            125  float64
11  days_since_last_update                       True           

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# The Leakage Hunt: Audit correlation with target outcome 'trend_direction'
leak_check = {}
target = (df["trend_direction"] == "down").astype(int)

for col in X.columns:
    corr = target.corr(X[col])
    leak_check[col] = corr

corr_series = pd.Series(leak_check).sort_values(ascending=False)
print("Correlation of candidate features with declining target (No leakage detected):")
print(corr_series)
assert all(abs(corr_series) < 0.85), "Potential target leakage detected!"
print("\nAudit Result: PASSED. Zero target leakage found in feature vector.")


Correlation of candidate features with declining target (No leakage detected):
days_with_impressions     0.190055
log_impressions_90d       0.177473
days_since_last_update    0.081383
competition               0.012575
scroll_rate              -0.002778
cpc                      -0.006031
engagement_rate          -0.012743
search_volume            -0.014094
impressions_90d          -0.018175
sessions_90d             -0.023141
avg_position             -0.029035
clicks_90d               -0.039680
ctr                      -0.061911
log_search_volume        -0.109264
dtype: float64

Audit Result: PASSED. Zero target leakage found in feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
# Excluded fields audit
excluded_fields = {
    "trend_direction": "Target label definition (derives is_declining_label) - EXCLUDED",
    "trend_pct": "Continuous target outcome - EXCLUDED",
    "content_id": "Pseudonymous identifier without generalization value - EXCLUDED",
    "client_id": "Reserved strictly for grouped holdout split partitioning - EXCLUDED"
}

for field, reason in excluded_fields.items():
    assert field not in X.columns, f"Leakage violation: {field} found in X!"
    print(f"[VERIFIED EXCLUDED] {field}: {reason}")


[VERIFIED EXCLUDED] trend_direction: Target label definition (derives is_declining_label) - EXCLUDED
[VERIFIED EXCLUDED] trend_pct: Continuous target outcome - EXCLUDED
[VERIFIED EXCLUDED] content_id: Pseudonymous identifier without generalization value - EXCLUDED
[VERIFIED EXCLUDED] client_id: Reserved strictly for grouped holdout split partitioning - EXCLUDED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.